# CyberSafe USeP — Upgraded Analysis Notebook
**Group 2 | CS Research Methods | USeP CIC**

This notebook conducts a full secondary qualitative analysis of the CyberSafe USeP dataset including:
- **Thematic Analysis** — expanded coding, uncategorized deep dive, theme development
- **Content Analysis** — word frequency tracking across 5 term groups
- **Role and Office Comparison** — students vs personnel, technical vs non-technical
- **Cross-Reference** — themes vs word frequency evidence
- **Framework Input Summary** — data-justified components for the training framework

---

## STEP 0 — Install and Import

In [ ]:
# Install required libraries
!pip install python-docx openpyxl pandas matplotlib seaborn wordcloud -q

In [ ]:
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from docx import Document
from collections import Counter, defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

print('All libraries loaded.')

---
## STEP 1 — Load All Three Datasets

In [ ]:
# ============================================================
# FILE PATHS — change these to match your local file locations
# ============================================================
SURVEY_PATH    = 'cybersafe_data.xlsx'
TRANSCRIPT_PATH = 'Minutes_1_.docx'
NVIVO_PATH     = 'results3.xlsx'

# --- Load Survey ---
wb = openpyxl.load_workbook(SURVEY_PATH, read_only=True)
ws = wb['Form Responses 2']
survey_rows = list(ws.iter_rows(values_only=True))
survey_headers = survey_rows[0]
survey_data = survey_rows[1:]  # exclude header
print(f'Survey loaded: {len(survey_data)} respondents, {len(survey_headers)} columns')

# --- Load Transcript ---
doc = Document(TRANSCRIPT_PATH)
transcript_paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
transcript_full = ' '.join(transcript_paragraphs)
print(f'Transcript loaded: {len(transcript_full.split())} words, {len(transcript_paragraphs)} paragraphs')

# --- Load NVivo Results ---
wb2 = openpyxl.load_workbook(NVIVO_PATH, read_only=True)
ws2 = wb2['Sheet2']
nvivo_rows = list(ws2.iter_rows(values_only=True))
nvivo_data = nvivo_rows[1:]  # exclude header
print(f'NVivo results loaded: {len(nvivo_data)} rows')

---
## STEP 2 — Demographics Overview

In [ ]:
# Extract demographic info
roles   = [row[2] for row in survey_data]
depts   = [row[3] for row in survey_data]
years   = [row[4] for row in survey_data]
ages    = [row[5] for row in survey_data]

role_counts  = Counter(roles)
age_counts   = Counter(ages)
years_counts = Counter(years)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('CyberSafe USeP — Respondent Demographics (n=600)', fontsize=15, fontweight='bold')

# Role distribution
axes[0].barh(list(role_counts.keys()), list(role_counts.values()),
             color=['#2E4057','#048A81','#54C6EB','#EF946C'])
axes[0].set_title('By Role')
axes[0].set_xlabel('Count')
for i, (k, v) in enumerate(role_counts.items()):
    axes[0].text(v + 3, i, str(v), va='center', fontsize=11)

# Age distribution
age_order = ['Under 25', '25 to 34', '35 to 44', '45 to 54', '55 and over']
age_vals  = [age_counts.get(a, 0) for a in age_order]
axes[1].barh(age_order, age_vals, color='#048A81')
axes[1].set_title('By Age Range')
axes[1].set_xlabel('Count')
for i, v in enumerate(age_vals):
    axes[1].text(v + 3, i, str(v), va='center', fontsize=11)

# Years in USeP
yr_order = ['Less than 5 years', '5 to 10 years', '11 to 15 years', 'More than 15 years']
yr_vals  = [years_counts.get(y, 0) for y in yr_order]
axes[2].barh(yr_order, yr_vals, color='#54C6EB')
axes[2].set_title('By Years in USeP')
axes[2].set_xlabel('Count')
for i, v in enumerate(yr_vals):
    axes[2].text(v + 3, i, str(v), va='center', fontsize=11)

plt.tight_layout()
plt.savefig('demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Demographics chart saved.')

---
## STEP 3 — Extract All Open-Ended Responses

In [ ]:
# Open-ended column indices (0-based)
OE_COLS = {
    'data_privacy_perception': 6,
    'cybersec_perception':     7,
    'worries_privacy':         32,
    'worries_cybersec':        33,
    'tools_needed':            49,
    'additional_comments':     52,
}

# Words that indicate a non-response
SKIP_WORDS = {'none', 'n/a', 'na', 'nothing', 'no', 'no comment', 'no suggestion',
              'not applicable', 'i dont know', "i don't know", 'no worries',
              'no idea', 'no problem', '.', '-', 'null', 'nope', 'nothing else',
              'none.', 'none so far', 'nothing for now', 'good'}

def is_meaningful(text):
    if not text or not isinstance(text, str):
        return False
    cleaned = text.strip().lower()
    if cleaned in SKIP_WORDS or len(cleaned) < 5:
        return False
    return True

# Build records
records = []
for row in survey_data:
    role = str(row[2]) if row[2] else 'Unknown'
    dept = str(row[3]) if row[3] else 'Unknown'
    entry = {'role': role, 'dept': dept}
    for col_name, col_idx in OE_COLS.items():
        text = str(row[col_idx]).strip() if row[col_idx] else ''
        entry[col_name] = text if is_meaningful(text) else ''
    records.append(entry)

df = pd.DataFrame(records)

# Combined text per respondent (all open-ended responses merged)
df['all_text'] = df[list(OE_COLS.keys())].apply(
    lambda x: ' '.join([v for v in x if v]), axis=1
)

print(f'Total records: {len(df)}')
for col in OE_COLS:
    filled = df[col].apply(lambda x: len(x) > 0).sum()
    print(f'  {col}: {filled} meaningful responses ({filled/6:.1f}%)')

---
## STEP 4 — EXPANDED THEMATIC ANALYSIS
### 4A — Comprehensive Coding Dictionary (Upgraded)

In [ ]:
# ============================================================
# EXPANDED CODING DICTIONARY
# Organized into 4 major theme groups with subthemes
# ============================================================

THEME_CODES = {
    # ---- THEME 1: SURFACE-LEVEL THREAT AWARENESS ----
    'Awareness of Phishing/Scams': [
        'phishing', 'phish', 'scam', 'spam', 'suspicious link',
        'fake email', 'email scam', 'fake message', 'suspicious email',
        'suspicious message', 'fake account'
    ],
    'Hacking as Primary Threat': [
        'hack', 'hacked', 'hacking', 'hacker', 'brute force',
        'unauthorized access', 'account takeover', 'hijack'
    ],
    'Virus/Malware Mention': [
        'virus', 'malware', 'ransomware', 'trojan', 'spyware',
        'worm', 'infected', 'infection', 'malicious software'
    ],
    'Data Breach Concern': [
        'breach', 'data breach', 'leak', 'leakage', 'exposed',
        'compromised data', 'data exposure', 'stolen data'
    ],

    # ---- THEME 2: EMOTIONAL RESPONSES ----
    'Fear of Hacking/Leaks': [
        'scared', 'fear', 'afraid', 'terrified', 'frightened',
        'dread', 'frightening', 'i am scared', 'i am afraid'
    ],
    'Worry and Anxiety': [
        'worry', 'worried', 'concern', 'concerned', 'anxious',
        'anxiety', 'nervous', 'stress', 'stressed', 'uneasy'
    ],
    'Feeling Overwhelmed': [
        'overwhelming', 'overwhelmed', 'complicated', 'complex',
        'confusing', 'confused', 'too much', 'hard to understand'
    ],
    'Emotional Distress': [
        'helpless', 'unsafe', 'vulnerable', 'powerless',
        'at risk', 'not safe', 'in danger', 'cannot protect'
    ],

    # ---- THEME 3: INSTITUTIONAL FACTORS ----
    'Institutional Inadequacy': [
        'lack of', 'not enough', 'insufficient', 'inadequate',
        'should improve', 'needs to improve', 'needs more',
        'not doing enough', 'poor', 'weak system', 'no helpdesk',
        'no response team', 'no clear policy', 'no framework'
    ],
    'Budget Constraints': [
        'budget', 'funding', 'fund', 'expensive', 'cost',
        'resources', 'limited resources', 'no money', 'state university'
    ],
    'Request for Training': [
        'training', 'seminar', 'workshop', 'orientation', 'educate',
        'education', 'awareness program', 'learn more', 'teach',
        'capacity building', 'self-paced', 'module', 'webinar'
    ],
    'Request for Policy/Support': [
        'policy', 'policies', 'guideline', 'guidelines', 'enforce',
        'mandate', 'require', 'helpdesk', 'hotline', 'help desk',
        'reporting mechanism', 'response team', 'roadmap'
    ],
    'Passive Trust in USeP': [
        'trust usep', 'trust the university', 'confident in usep',
        'university is doing well', 'university is safe',
        'usep is protecting', 'they are doing their best',
        'i believe usep', 'usep is okay', 'university handles it'
    ],

    # ---- THEME 4: KNOWLEDGE AND PRACTICE ----
    'Uncertainty/Lack of Knowledge': [
        'not aware', 'unaware', 'do not know', "don't know",
        'not familiar', 'unfamiliar', 'no knowledge', 'not sure',
        'unsure', 'i have no idea', 'not fully aware'
    ],
    'Password and Account Security': [
        'password', 'passwords', 'strong password', 'change password',
        'two-factor', '2fa', 'two factor', 'authentication',
        'account security', 'login', 'credentials'
    ],
    'Platform/Tool Dependency': [
        'messenger', 'facebook', 'google drive', 'gmail', 'email',
        'zoom', 'microsoft teams', 'social media', 'antivirus',
        'software', 'app', 'application', 'platform'
    ],
    'Personal Responsibility Awareness': [
        'responsible', 'responsibility', 'individual', 'personal',
        'each person', 'everyone should', 'we should', 'users should',
        'shared responsibility', 'commitment'
    ],
    'Risky Behavior Acknowledgment': [
        'share account', 'sharing account', 'credential sharing',
        'not logging out', 'forgot to logout', 'weak password',
        'write password', 'password on paper', 'convenience'
    ],
}

print(f'Coding dictionary loaded: {len(THEME_CODES)} codes')
print('Codes defined:')
for code in THEME_CODES:
    print(f'  - {code} ({len(THEME_CODES[code])} keywords)')

### 4B — Apply Coding to All Text Sources

In [ ]:
def apply_codes(text, code_dict):
    """Returns list of codes that match the given text."""
    if not text or not isinstance(text, str):
        return []
    text_lower = text.lower()
    matched = []
    for code, keywords in code_dict.items():
        for kw in keywords:
            if kw.lower() in text_lower:
                matched.append(code)
                break
    return matched if matched else ['Uncategorized']


# Apply to survey open-ended responses
df['codes'] = df['all_text'].apply(lambda t: apply_codes(t, THEME_CODES))

# Explode codes for frequency counting
code_series = df['codes'].explode()
survey_code_counts = Counter(code_series.tolist())

# Apply to transcript
transcript_codes = []
for para in transcript_paragraphs:
    codes = apply_codes(para, THEME_CODES)
    transcript_codes.extend(codes)
transcript_code_counts = Counter(transcript_codes)

# Apply to NVivo quotes
nvivo_codes = []
for row in nvivo_data:
    quote_text = str(row[2]) if row[2] else ''
    codes = apply_codes(quote_text, THEME_CODES)
    nvivo_codes.extend(codes)
nvivo_code_counts = Counter(nvivo_codes)

# COMBINED across all three sources
all_code_counts = Counter()
for c in [survey_code_counts, transcript_code_counts, nvivo_code_counts]:
    all_code_counts.update(c)

print('Code frequencies across ALL sources:')
for code, count in all_code_counts.most_common():
    print(f'  {code}: {count}')

### 4C — Upgraded Frequency Chart (All Codes)

In [ ]:
# Color map by theme group
THEME_GROUPS = {
    'Awareness of Phishing/Scams':      '#E07A5F',
    'Hacking as Primary Threat':         '#E07A5F',
    'Virus/Malware Mention':             '#E07A5F',
    'Data Breach Concern':               '#E07A5F',
    'Fear of Hacking/Leaks':             '#F2CC8F',
    'Worry and Anxiety':                 '#F2CC8F',
    'Feeling Overwhelmed':               '#F2CC8F',
    'Emotional Distress':                '#F2CC8F',
    'Institutional Inadequacy':          '#3D405B',
    'Budget Constraints':                '#3D405B',
    'Request for Training':              '#3D405B',
    'Request for Policy/Support':        '#3D405B',
    'Passive Trust in USeP':             '#3D405B',
    'Uncertainty/Lack of Knowledge':     '#81B29A',
    'Password and Account Security':     '#81B29A',
    'Platform/Tool Dependency':          '#81B29A',
    'Personal Responsibility Awareness': '#81B29A',
    'Risky Behavior Acknowledgment':     '#81B29A',
    'Uncategorized':                     '#BBBBBB',
}

sorted_codes = all_code_counts.most_common()
labels = [c[0] for c in sorted_codes]
values = [c[1] for c in sorted_codes]
colors = [THEME_GROUPS.get(l, '#BBBBBB') for l in labels]

fig, ax = plt.subplots(figsize=(14, 10))
bars = ax.barh(labels, values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Number of Mentions', fontsize=12)
ax.set_title('Frequency of Thematic Codes Across All Data Sources\n(Survey + Transcript + NVivo)',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val),
            va='center', fontsize=10)

# Legend
legend_patches = [
    mpatches.Patch(color='#E07A5F', label='Theme 1: Surface Threat Awareness'),
    mpatches.Patch(color='#F2CC8F', label='Theme 2: Emotional Responses'),
    mpatches.Patch(color='#3D405B', label='Theme 3: Institutional Factors'),
    mpatches.Patch(color='#81B29A', label='Theme 4: Knowledge and Practice'),
    mpatches.Patch(color='#BBBBBB', label='Uncategorized'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('thematic_codes_expanded.png', dpi=150, bbox_inches='tight')
plt.show()
print('Expanded thematic chart saved.')

---
## STEP 5 — UNCATEGORIZED DEEP DIVE

In [ ]:
# Extract all uncategorized responses
uncategorized_df = df[df['codes'].apply(lambda c: c == ['Uncategorized'])].copy()
print(f'Total uncategorized respondents: {len(uncategorized_df)}')
print(f'Percentage of total: {len(uncategorized_df)/len(df)*100:.1f}%')

# Classify uncategorized into sub-groups
UNCAT_SUBCODES = {
    'Truly Blank or Non-Response': [
        'none', 'n/a', 'na', 'nothing', 'no', 'no comment',
        'not applicable', "i don't know", 'i dont know', 'no worries',
        'nothing to add', 'no idea', '.', '-', 'null'
    ],
    'Vague Positive Statement': [
        'very good', 'good', 'great', 'excellent', 'ok', 'okay',
        'fine', 'effective', 'satisfactory', 'sufficient', '8/10', '9/10', '10/10'
    ],
    'General Improvement Request': [
        'improve', 'better', 'enhance', 'strengthen', 'develop',
        'upgrade', 'update', 'more', 'increase', 'boost'
    ],
    'Specific Tool Request': [
        'antivirus', 'vpn', 'firewall', 'software', 'tool',
        'system', 'app', 'application', 'platform', 'install'
    ],
    'Awareness Campaign Request': [
        'campaign', 'awareness', 'spread', 'inform', 'announce',
        'communicate', 'disseminate', 'broadcast', 'publicize'
    ],
    'Student Perspective Only': [
        'as a student', 'we students', 'student body',
        'student council', 'for students', 'student awareness'
    ],
    'Reference to Specific Incident': [
        'facebook', 'hacked page', 'our office', 'our department',
        'last year', 'incident', 'happened to us', 'our registrar'
    ],
}

uncat_sub_codes = []
for _, row in uncategorized_df.iterrows():
    sub = apply_codes(row['all_text'], UNCAT_SUBCODES)
    uncat_sub_codes.extend(sub)

uncat_sub_counts = Counter(uncat_sub_codes)
print('\nUncategorized Sub-Code Breakdown:')
for code, count in uncat_sub_counts.most_common():
    pct = count / len(uncategorized_df) * 100
    print(f'  {code}: {count} ({pct:.1f}%)')

In [ ]:
# Visualize uncategorized sub-codes
sub_labels = [c[0] for c in uncat_sub_counts.most_common()]
sub_values = [c[1] for c in uncat_sub_counts.most_common()]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(sub_labels, sub_values,
               color=['#C2185B','#7B1FA2','#1565C0','#00695C','#E65100','#558B2F','#4E342E'][:len(sub_labels)])
ax.set_xlabel('Count')
ax.set_title('Deep Dive: What is Inside the Uncategorized Pile?', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, sub_values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center')
plt.tight_layout()
plt.savefig('uncategorized_deep_dive.png', dpi=150, bbox_inches='tight')
plt.show()

# Show sample uncategorized responses
print('\nSample uncategorized responses:')
samples = uncategorized_df['all_text'].dropna().head(20)
for i, s in enumerate(samples):
    if len(s) > 10:
        print(f'  [{i+1}] {s[:120]}')

---
## STEP 6 — CONTENT ANALYSIS: Word Frequency Tracking

In [ ]:
# ============================================================
# TERM GROUPS FOR CONTENT ANALYSIS
# ============================================================
TERM_GROUPS = {
    'Group A — Surface Threats': [
        'email', 'phishing', 'hacking', 'hacked', 'virus',
        'malware', 'scam', 'ransomware', 'suspicious', 'fake', 'breach', 'leak'
    ],
    'Group B — Security Practices': [
        'password', 'two-factor', '2fa', 'antivirus', 'update',
        'backup', 'logout', 'lock', 'privacy settings', 'authentication'
    ],
    'Group C — Advanced Concepts': [
        'encryption', 'access control', 'incident response', 'vulnerability',
        'firewall', 'penetration', 'network security', 'zero trust',
        'data governance', 'cybersecurity framework', 'intrusion'
    ],
    'Group D — Emotional Terms': [
        'scared', 'fear', 'worried', 'overwhelming', 'unsafe',
        'concerned', 'confused', 'helpless', 'cautious', 'anxious'
    ],
    'Group E — Institutional Terms': [
        'training', 'seminar', 'policy', 'guidelines', 'budget',
        'helpdesk', 'orientation', 'reporting', 'sdmd', 'framework'
    ],
}

# Combine all text sources
survey_all_text = ' '.join(df['all_text'].tolist()).lower()
transcript_text = transcript_full.lower()
nvivo_all_text  = ' '.join([str(r[2]) for r in nvivo_data if r[2]]).lower()
combined_text   = survey_all_text + ' ' + transcript_text + ' ' + nvivo_all_text

def count_term(term, text):
    return len(re.findall(r'\b' + re.escape(term.lower()) + r'\b', text))

# Build frequency table
freq_results = []
for group, terms in TERM_GROUPS.items():
    for term in terms:
        total   = count_term(term, combined_text)
        survey  = count_term(term, survey_all_text)
        trans   = count_term(term, transcript_text)
        freq_results.append({
            'Group': group,
            'Term': term,
            'Total': total,
            'Survey': survey,
            'Transcript': trans,
        })

freq_df = pd.DataFrame(freq_results)
print('Word Frequency Results:')
print(freq_df.to_string(index=False))

In [ ]:
# Group totals and Surface vs Advanced ratio
group_totals = freq_df.groupby('Group')['Total'].sum().reset_index()
group_totals = group_totals.sort_values('Total', ascending=False)

group_a_total = group_totals[group_totals['Group'].str.startswith('Group A')]['Total'].sum()
group_c_total = group_totals[group_totals['Group'].str.startswith('Group C')]['Total'].sum()
ratio = group_a_total / group_c_total if group_c_total > 0 else float('inf')

print(f'Group A (Surface Threats) total mentions: {group_a_total}')
print(f'Group C (Advanced Concepts) total mentions: {group_c_total}')
print(f'Surface-to-Advanced Ratio: {ratio:.1f}x')
print()
print('This ratio tells us: for every 1 advanced cybersecurity concept mentioned,')
print(f'stakeholders mention surface-level threats {ratio:.1f} times.')
print('This is the quantitative proof of surface-level awareness.')

# Visualize group totals
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Content Analysis — Word Frequency by Group', fontsize=14, fontweight='bold')

group_colors = ['#E07A5F','#81B29A','#3D405B','#F2CC8F','#54C6EB']
axes[0].bar(group_totals['Group'].str[:12], group_totals['Total'],
            color=group_colors[:len(group_totals)])
axes[0].set_title('Total Mentions per Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Surface vs Advanced side-by-side
axes[1].bar(['Group A\n(Surface Threats)', 'Group C\n(Advanced Concepts)'],
            [group_a_total, group_c_total],
            color=['#E07A5F', '#3D405B'], width=0.4)
axes[1].set_title(f'Surface vs Advanced Ratio: {ratio:.1f}x')
axes[1].set_ylabel('Count')
for i, v in enumerate([group_a_total, group_c_total]):
    axes[1].text(i, v + 2, str(v), ha='center', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('content_analysis_groups.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top terms per group
print('Top 5 terms per group:')
for group in TERM_GROUPS:
    top = freq_df[freq_df['Group'] == group].nlargest(5, 'Total')
    print(f'\n{group}:')
    for _, row in top.iterrows():
        print(f'  "{row["Term"]}": {row["Total"]} total (Survey: {row["Survey"]}, Transcript: {row["Transcript"]})')

---
## STEP 7 — ROLE-BASED COMPARISON

In [ ]:
# Code frequency breakdown by role
role_code_matrix = defaultdict(Counter)
for _, row in df.iterrows():
    role = row['role']
    for code in row['codes']:
        role_code_matrix[role][code] += 1

roles_list = ['Student', 'Staff', 'Faculty', 'Administrator']
all_codes_list = [c for c in all_code_counts if c != 'Uncategorized']

# Build matrix
matrix_data = {}
for role in roles_list:
    matrix_data[role] = [role_code_matrix[role].get(c, 0) for c in all_codes_list]

matrix_df = pd.DataFrame(matrix_data, index=all_codes_list)

# Heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(matrix_df, annot=True, fmt='d', cmap='YlOrRd',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Frequency'})
ax.set_title('Thematic Code Frequency by Respondent Role', fontsize=14, fontweight='bold')
ax.set_xlabel('Role')
ax.set_ylabel('Code')
plt.tight_layout()
plt.savefig('role_code_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 codes per role:')
for role in roles_list:
    top3 = role_code_matrix[role].most_common(3)
    print(f'  {role}: {top3}')

In [ ]:
# Word frequency comparison across roles — Group A vs Group C
role_group_a = {}
role_group_c = {}

for role in roles_list:
    role_text = ' '.join([
        row['all_text'] for _, row in df[df['role'] == role].iterrows()
    ]).lower()
    ga = sum(count_term(t, role_text) for t in TERM_GROUPS['Group A — Surface Threats'])
    gc = sum(count_term(t, role_text) for t in TERM_GROUPS['Group C — Advanced Concepts'])
    role_group_a[role] = ga
    role_group_c[role] = gc

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(roles_list))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], [role_group_a[r] for r in roles_list],
               width, label='Group A: Surface Threats', color='#E07A5F')
bars2 = ax.bar([i + width/2 for i in x], [role_group_c[r] for r in roles_list],
               width, label='Group C: Advanced Concepts', color='#3D405B')
ax.set_xticks(x)
ax.set_xticklabels(roles_list)
ax.set_ylabel('Term Frequency')
ax.set_title('Surface vs Advanced Term Use — By Respondent Role', fontsize=13, fontweight='bold')
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('role_surface_vs_advanced.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 8 — OFFICE TYPE COMPARISON (Transcript)
### Technical vs Non-Technical Offices

In [ ]:
# Map offices from transcript into technical vs non-technical
TECHNICAL_OFFICES    = ['SDMD', 'CIC']
NON_TECHNICAL_OFFICES = ['Finance', 'HRMD', 'OUR', 'RDE', 'Procurement',
                         'UAGC', 'OSAS', 'ULRC', 'OLA', 'USO-URO',
                         'UDPO', 'Publication']

def classify_paragraph(text):
    text_upper = text.upper()
    for office in TECHNICAL_OFFICES:
        if office.upper() in text_upper:
            return 'Technical'
    for office in NON_TECHNICAL_OFFICES:
        if office.upper() in text_upper:
            return 'Non-Technical'
    return 'General'

tech_paragraphs    = []
nontech_paragraphs = []

current_office_type = 'General'
for para in transcript_paragraphs:
    classification = classify_paragraph(para)
    if classification != 'General':
        current_office_type = classification
    if current_office_type == 'Technical':
        tech_paragraphs.append(para)
    elif current_office_type == 'Non-Technical':
        nontech_paragraphs.append(para)

tech_text    = ' '.join(tech_paragraphs).lower()
nontech_text = ' '.join(nontech_paragraphs).lower()

print(f'Technical office paragraphs: {len(tech_paragraphs)}')
print(f'Non-technical office paragraphs: {len(nontech_paragraphs)}')

# Compare term group usage
print('\nTerm Group Frequency Comparison:')
print(f'{"Group":<35} {"Technical":>12} {"Non-Technical":>15}')
print('-' * 65)
for group, terms in TERM_GROUPS.items():
    tech_count    = sum(count_term(t, tech_text) for t in terms)
    nontech_count = sum(count_term(t, nontech_text) for t in terms)
    print(f'{group[:35]:<35} {tech_count:>12} {nontech_count:>15}')

In [ ]:
# Visual comparison
groups = list(TERM_GROUPS.keys())
tech_vals    = [sum(count_term(t, tech_text) for t in TERM_GROUPS[g]) for g in groups]
nontech_vals = [sum(count_term(t, nontech_text) for t in TERM_GROUPS[g]) for g in groups]

short_groups = [g.split('—')[0].strip() for g in groups]

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(groups))
ax.bar([i - 0.2 for i in x], tech_vals, 0.4, label='Technical Offices (SDMD, CIC)', color='#2E4057')
ax.bar([i + 0.2 for i in x], nontech_vals, 0.4, label='Non-Technical Offices', color='#E07A5F')
ax.set_xticks(x)
ax.set_xticklabels(short_groups, rotation=20, ha='right')
ax.set_ylabel('Term Frequency')
ax.set_title('Technical vs Non-Technical Offices\nTerm Group Usage in Interview Transcript',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('office_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Emotional term breakdown per office type
print('\nEmotional term mentions:')
for term in TERM_GROUPS['Group D — Emotional Terms']:
    t = count_term(term, tech_text)
    n = count_term(term, nontech_text)
    if t + n > 0:
        print(f'  "{term}": Technical={t}, Non-Technical={n}')

---
## STEP 9 — CROSS-REFERENCE: Themes x Word Frequency

In [ ]:
# For each major theme code, check what word groups appear most in those responses
MAJOR_CODES = [
    'Institutional Inadequacy',
    'Fear of Hacking/Leaks',
    'Request for Training',
    'Request for Policy/Support',
    'Awareness of Phishing/Scams',
    'Worry and Anxiety',
    'Uncertainty/Lack of Knowledge',
    'Emotional Distress',
    'Passive Trust in USeP',
]

print('Cross-Reference: What word groups dominate each theme?')
print('=' * 70)
for code in MAJOR_CODES:
    code_responses = df[df['codes'].apply(lambda c: code in c)]['all_text']
    code_text = ' '.join(code_responses.tolist()).lower()
    print(f'\nTheme: {code} ({len(code_responses)} responses)')
    for group, terms in TERM_GROUPS.items():
        count = sum(count_term(t, code_text) for t in terms)
        short = group.split('—')[1].strip() if '—' in group else group
        bar = '|' * min(count, 40)
        print(f'  {short:<25}: {bar} {count}')

---
## STEP 10 — THEME DEVELOPMENT SUMMARY
### Mapping Codes to Final Themes

In [ ]:
# Final theme groupings based on analysis
FINAL_THEMES = {
    'Theme 1: Surface-Level Cybersecurity Literacy': [
        'Awareness of Phishing/Scams',
        'Hacking as Primary Threat',
        'Virus/Malware Mention',
        'Data Breach Concern',
        'Uncertainty/Lack of Knowledge',
    ],
    'Theme 2: Fear-Driven Security Perception': [
        'Fear of Hacking/Leaks',
        'Worry and Anxiety',
        'Feeling Overwhelmed',
        'Emotional Distress',
    ],
    'Theme 3: Institutional Dependence and Inadequacy': [
        'Institutional Inadequacy',
        'Budget Constraints',
        'Passive Trust in USeP',
        'Request for Policy/Support',
    ],
    'Theme 4: Practice Gap and Behavioral Risk': [
        'Risky Behavior Acknowledgment',
        'Password and Account Security',
        'Platform/Tool Dependency',
        'Personal Responsibility Awareness',
    ],
    'Theme 5: Demand for Structured Training': [
        'Request for Training',
    ],
}

theme_totals = {}
for theme, codes in FINAL_THEMES.items():
    total = sum(all_code_counts.get(c, 0) for c in codes)
    theme_totals[theme] = total

print('Final Theme Frequency Summary:')
for theme, total in sorted(theme_totals.items(), key=lambda x: x[1], reverse=True):
    print(f'  {theme}: {total} total code mentions')

# Final theme chart
fig, ax = plt.subplots(figsize=(12, 6))
short_themes = [t.split(':')[1].strip() for t in theme_totals.keys()]
t_values = list(theme_totals.values())
theme_colors = ['#E07A5F','#F2CC8F','#3D405B','#81B29A','#54C6EB']
bars = ax.bar(short_themes, t_values, color=theme_colors[:len(short_themes)])
ax.set_ylabel('Total Code Mentions')
ax.set_title('Final Theme Frequency — CyberSafe USeP Re-Analysis', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, t_values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5, str(val),
            ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('final_themes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 11 — FRAMEWORK INPUT SUMMARY
### Data-Justified Components for the Training Framework

In [ ]:
print('=' * 70)
print('FRAMEWORK INPUT SUMMARY')
print('CyberSafe USeP — Cybersecurity Literacy Training Framework')
print('=' * 70)

print('''
COMPONENT 1 — TARGET STAKEHOLDERS AND IDENTIFIED GAPS
------------------------------------------------------
STUDENTS (n=393, 65.5% of sample):
  - Dominant codes: Institutional Inadequacy, Fear of Hacking/Leaks
  - Surface-level threat vocabulary dominates (email, hacking, breach)
  - Near-zero use of advanced cybersecurity terminology
  - Most responses show passive trust rather than informed confidence
  - Training gap: Basic cybersecurity hygiene + threat literacy

STAFF (n=126, 21% of sample):
  - Dominant codes: Request for Training, Institutional Inadequacy
  - Directly responsible for sensitive office data daily
  - Interview data shows real incidents (phishing, ransomware, account leaks)
  - Training gap: Role-specific threat response + incident reporting

FACULTY (n=74, 12.3% of sample):
  - Dominant codes: Institutional Inadequacy, Request for Policy/Support
  - Handle student data and research records
  - Training gap: Data handling policies + research data protection

ADMINISTRATORS (n=7, 1.2% of sample):
  - Small sample but critical for policy implementation
  - Training gap: Governance-level cybersecurity oversight

TECHNICAL OFFICES (SDMD, CIC):
  - Higher use of technical vocabulary
  - Aware of shared responsibility but understaffed
  - Training gap: User communication + policy enforcement skills

NON-TECHNICAL OFFICES (Finance, HRMD, OUR, RDE, etc.):
  - Emotional terms dominant (scared, worried, overwhelmed)
  - Limited use of technical vocabulary
  - Real incidents documented but poor incident response knowledge
  - Training gap: Foundational threat awareness + behavioral practice
''')

print('''
COMPONENT 2 — TRAINING INTERVENTION PER GAP
--------------------------------------------
Level 1 (All stakeholders): Awareness campaigns — what cybersecurity is,
  what the real threats look like beyond phishing and email scams.

Level 2 (Students + Non-technical staff): Behavioral training — password
  hygiene, 2FA, logging out, not sharing credentials, spotting phishing.

Level 3 (Staff + Faculty): Role-based training — specific to the kind of
  data they handle and what incidents have actually happened in their office.

Level 4 (Administrators + SDMD): Policy and governance training — how to
  build enforceable policies, how to run incident response, how to cascade.
''')

print('''
COMPONENT 3 — EMOTIONAL BARRIERS TO ADDRESS FIRST
--------------------------------------------------
The data shows fear and overwhelm BEFORE lack of knowledge.
Any training framework must begin by normalizing cybersecurity —
making it feel manageable and personal rather than technical and scary.
Training sessions must open with real USeP incidents (not hypotheticals)
to make threats feel real but also to show that they are survivable.
''')

print('''
COMPONENT 4 — INSTITUTIONAL CONDITIONS REQUIRED
------------------------------------------------
Based on data: training alone will not work without these conditions:
1. A clear incident reporting channel (helpdesk or hotline)
2. Leadership mandate for 2FA and password policies
3. Budget allocation for at least annual cybersecurity training
4. A designated response team for cybersecurity incidents
5. Cascade system to ensure training reaches all staff, not just representatives
''')

print('''
COMPONENT 5 — WHAT SUCCESS LOOKS LIKE
--------------------------------------
After framework implementation, USeP stakeholders should be able to:
- Name at least 3 cybersecurity threats beyond phishing and email scams
- Consistently use 2FA and strong passwords without being reminded
- Know exactly where and how to report a cybersecurity incident
- Describe cybersecurity as a personal responsibility, not just an IT issue
- Show low-to-moderate use of emotional distress language in responses
  (replaced by cautious, practical, action-oriented language)
''')

print('Framework Input Summary complete.')

---
## STEP 12 — EXPORT ALL RESULTS

In [ ]:
# Export frequency tables to Excel
with pd.ExcelWriter('CyberSafe_Analysis_Results.xlsx') as writer:

    # Sheet 1: All code frequencies
    code_df = pd.DataFrame(all_code_counts.most_common(),
                           columns=['Code', 'Total Mentions'])
    code_df.to_excel(writer, sheet_name='Thematic Codes', index=False)

    # Sheet 2: Word frequency
    freq_df.to_excel(writer, sheet_name='Word Frequency', index=False)

    # Sheet 3: Role comparison
    role_summary = []
    for role in roles_list:
        top_codes = role_code_matrix[role].most_common(5)
        role_summary.append({'Role': role, 'Top Codes': str(top_codes)})
    pd.DataFrame(role_summary).to_excel(writer, sheet_name='Role Comparison', index=False)

    # Sheet 4: Final themes
    theme_summary = [{'Theme': k, 'Total Mentions': v}
                     for k, v in theme_totals.items()]
    pd.DataFrame(theme_summary).to_excel(writer, sheet_name='Final Themes', index=False)

print('Results exported to CyberSafe_Analysis_Results.xlsx')
print('Charts saved as PNG files.')
print()
print('Files generated:')
print('  - demographics.png')
print('  - thematic_codes_expanded.png')
print('  - uncategorized_deep_dive.png')
print('  - content_analysis_groups.png')
print('  - role_code_heatmap.png')
print('  - role_surface_vs_advanced.png')
print('  - office_comparison.png')
print('  - final_themes.png')
print('  - CyberSafe_Analysis_Results.xlsx')

---
## STEP 13 — CLAUDE API PROMPTS (Copy-Paste Ready)
Run this cell to print all prompts you can paste into another Claude chat for deeper analysis.

In [ ]:
# Print ready-to-use Claude prompts based on current findings

code_summary = '\n'.join([f'{c}: {v}' for c, v in all_code_counts.most_common(15)])
group_summary = '\n'.join([f'{g}: {sum(count_term(t, combined_text) for t in TERM_GROUPS[g])}'
                           for g in TERM_GROUPS])

print('=' * 60)
print('PROMPT 1 — Theme Narrative Writing')
print('=' * 60)
print(f'''
I am analyzing cybersecurity awareness data from the University of
Southeastern Philippines. My thematic analysis produced these code
frequencies across 600 survey responses and interview transcripts:

{code_summary}

And my content analysis produced these word group frequencies:
{group_summary}

Based on this, write a 5-paragraph academic discussion section that:
1. Describes what these frequencies reveal about cybersecurity literacy at USeP
2. Connects the dominant codes to the awareness-practice gap
3. Explains what the Surface-to-Advanced term ratio means
4. Discusses the emotional dimension shown in the data
5. Concludes with what this means for designing a training framework

Write at a high school reading level, no bullet points, plain academic prose.
''')

print('=' * 60)
print('PROMPT 2 — Framework Component Writing')
print('=' * 60)
print('''
Based on the thematic and content analysis findings above, write the
Framework Development section of our research paper. The section should:

1. Explain how each of our 5 themes justifies a specific framework component
2. Describe the training levels we propose (Awareness, Behavioral, Role-Based, Governance)
3. Explain how the framework addresses the emotional barriers identified in the data
4. Describe what institutional conditions USeP must provide for the framework to work
5. Explain how we adapted NIST SP 800-50, SETA, and the SANS Maturity Model
   to fit the specific USeP context

Write this as one coherent academic section, 4-6 paragraphs, high school grammar level.
''')

---
**End of CyberSafe USeP Upgraded Analysis Notebook**

*Group 2 | CS Research Methods | USeP CIC*

Charts and Excel results are saved in the same folder as this notebook.